# 📊 Multiple Linear Regression (MLR) Template
This notebook provides a robust, reusable framework for performing and validating a Multiple Linear Regression analysis using `statsmodels`.

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence

### Step 2: Load Data and Exploratory Data Analysis (EDA)

In [ ]:
# df = pd.read_csv('your_dataset.csv')

# Create dummy data for template execution
np.random.seed(42)
df = pd.DataFrame({
    'predictor1': np.random.rand(100) * 10,
    'predictor2': np.random.rand(100) * 5,
    'target': np.random.rand(100) * 20
})

display(df.head())
display(df.describe().T)
print("Null values:\n", df.isnull().sum())

### Step 3: Check for Linearity and Multicollinearity
Before fitting the model, ensure continuous predictors have a linear relationship with the target, and check that predictors are not highly correlated with each other.

In [ ]:
# 1. Pairplot
sns.pairplot(df)
plt.suptitle("Pairplot of Features", y=1.02)
plt.show()

# 2. Correlation Matrix
plt.figure(figsize=(6,4))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2g')
plt.title('Correlation Matrix')
plt.show()

# 3. Variance Inflation Factor (VIF)
# VIF > 5 or 10 indicates problematic multicollinearity
independent_vars = ['predictor1', 'predictor2']
X = df[independent_vars].values
for i in range(X.shape[1]):
    vif = variance_inflation_factor(X, i)
    print(f"VIF for {independent_vars[i]}: {vif:.2f}")

### Step 4: Fit the Model using Statsmodels

In [ ]:
target_var = 'target'
formula_str = f"{target_var} ~ " + " + ".join(independent_vars)

model = smf.ols(formula=formula_str, data=df)
fitted = model.fit()

print(fitted.summary())

### Step 5: Check Independence (Residuals vs Predictors)
Residuals should be randomly scattered around 0.

In [ ]:
fig, axs = plt.subplots(1, len(independent_vars), figsize=(12, 4))
fig.subplots_adjust(wspace=0.3)
for index, col in enumerate(independent_vars):
    axs[index].scatter(df[col], fitted.resid, color='blue', edgecolor='k')
    axs[index].axhline(0, color='red', linestyle='--', lw=2)
    axs[index].set_title(f"Residuals vs {col}")
    axs[index].grid(True)
plt.show()

### Step 6: Check Homoscedasticity
Variance of residuals should be constant. We verify visually and with the Breusch-Pagan test (p > 0.05 means homoscedasticity is met).

In [ ]:
# Visual Inspection
plt.figure(figsize=(6,4))
plt.scatter(fitted.fittedvalues, fitted.resid, color='purple', edgecolor='k')
plt.axhline(0, color='red', linestyle='--', lw=2)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted Values vs Residuals")
plt.grid(True)
plt.show()

# Breusch-Pagan Test
bp_test = sms.het_breuschpagan(fitted.resid, fitted.model.exog)
print(f"Breusch-Pagan LM Statistic: {bp_test[0]:.4f}")
print(f"Breusch-Pagan p-value: {bp_test[1]:.4f}")

### Step 7: Check Normality of Residuals
Residuals should follow a normal distribution. We verify with a histogram and Q-Q plot.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axs[0].hist(fitted.resid, bins=15, color='cyan', edgecolor='k')
axs[0].set_title("Histogram of Residuals")

# Q-Q Plot
qqplot(fitted.resid_pearson, line='45', ax=axs[1])
axs[1].set_title("Q-Q Plot")
plt.grid(True)
plt.show()

### Step 8: Outlier Detection (Cook's Distance)
Check if any single data point is severely skewing the model. Values > 1 indicate highly influential outliers.

In [ ]:
influence = OLSInfluence(fitted)
(c, p) = influence.cooks_distance

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(c)), c, marker='o', linestyle='-')
plt.xlabel('Observation Index')
plt.ylabel("Cook's Distance")
plt.title("Cook's Distance Plot")
plt.grid(True)
plt.show()